In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# User Story 2: Nghiên cứu Giám sát An toàn & Chẩn đoán Rủi ro Nhiệt (Safety & Risk Monitor)\n",
    "\n",
    "## 1. Bảng ánh xạ Khung thiết kế What - Why - How (Tamara Munzner)\n",
    "\n",
    "| Thành phần | Đặc tả kỹ thuật & Nghiệp vụ |\n",
    "| :--- | :--- |\n",
    "| **WHAT (Dữ liệu)** | - Chuỗi OT đối chiếu ngưỡng cảnh báo (85°C) và nguy hiểm (95°C).<br>- Danh sách các đợt sự cố liên tục, thời lượng quá nhiệt (Hours) và tích lũy độ-giờ (Degree-Hours).<br>- Tọa độ làm mịn phi tuyến LOESS và Z-score biến động tải. |\n",
    "| **WHY (Tác vụ trừu tượng)** | - **Detect Outlier & Anomaly**: Phát hiện tức thời các thời điểm vượt ngưỡng đỏ 95°C.<br>- **Locate & Summarize Extremes**: Định vị mật độ cụm sự cố theo dòng thời gian và lượng hóa áp lực nhiệt tích lũy.<br>- **Explore Root Cause**: Truy vết nhánh phụ tải gây quá nhiệt đột biến bằng Z-score. |\n",
    "| **HOW (Mã hóa thị giác & Kênh)** | - **Threshold Line Chart**: Tô vùng diện tích vượt ngưỡng kết hợp điểm Pop-out đỏ crimson.<br>- **Barcode / Rug Anomaly Strip**: Băng mã vạch định vị tức thời cụm sự cố.<br>- **Đường cong phi tuyến LOESS tiền tính toán**: Đánh giá phản ứng phi tuyến giữa các nhánh tải và OT. |\n",
    "\n",
    "---\n",
    "\n",
    "## 2. Danh mục Tiêu đề Hành động (Action Titles Catalog)\n",
    "1. **Action Title 1 (Threshold & Pop-out):** *\"Nhiệt độ dầu OT vi phạm ngưỡng cảnh báo 85°C tổng cộng 14 đợt trong mùa hè, đỉnh điểm chạm mốc nguy hiểm 98.2°C đe dọa nứt hỏng cách điện.\"*\n",
    "2. **Action Title 2 (Barcode Strip):** *\"Các sự cố quá nhiệt nguy hiểm tập trung dày đặc vào tháng 7 và tháng 8, hoàn toàn vắng bóng trong các tháng mùa đông.\"*\n",
    "3. **Action Title 3 (LOESS Regression):** *\"Mức gia tăng nhiệt độ dầu bắt đầu tăng tốc đột biến khi công suất tải Cao áp Hữu ích (HUFL) vượt mốc 40 MW.\"*"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "\n",
    "# Thiết lập đường dẫn dữ liệu tương đối\n",
    "BASE_DIR = os.path.abspath('..')\n",
    "ARTIFACTS_DIR = os.path.join(BASE_DIR, 'output', 'artifacts', 'ETTh1')\n",
    "\n",
    "# Nạp dữ liệu chuỗi thời gian, danh sách sự cố và dữ liệu LOESS tính sẵn\n",
    "clean_df = pd.read_parquet(os.path.join(ARTIFACTS_DIR, 'clean_ett.parquet'))\n",
    "breaches_df = pd.read_parquet(os.path.join(ARTIFACTS_DIR, 'thermal_breach_events.parquet'))\n",
    "loess_df = pd.read_parquet(os.path.join(ARTIFACTS_DIR, 'loess_curves.parquet'))\n",
    "\n",
    "print(f\"Tổng số đợt sự cố nhiệt độ dầu >= 85°C: {len(breaches_df)}\")\n",
    "display(breaches_df.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Khám phá 1: Chuỗi OT Vượt ngưỡng tích hợp Barcode Anomaly Strip\n",
    "fig_sec = make_subplots(\n",
    "    rows=2, cols=1, shared_xaxes=True, row_heights=[0.8, 0.2],\n",
    "    vertical_spacing=0.06,\n",
    "    subplot_titles=[\n",
    "        \"Action Title: Nhiệt độ dầu OT vi phạm ngưỡng cảnh báo 85°C tổng cộng 14 đợt trong mùa hè\",\n",
    "        \"Action Title: Các sự cố quá nhiệt nguy hiểm tập trung dày đặc vào tháng 7 và tháng 8\"\n",
    "    ]\n",
    ")\n",
    "\n",
    "# 1. Đường OT thực tế\n",
    "fig_sec.add_trace(\n",
    "    go.Scatter(\n",
    "        x=clean_df['date'], y=clean_df['OT'],\n",
    "        mode='lines', line=dict(color='#1E3A8A', width=1.5),\n",
    "        name='Nhiệt độ dầu OT'\n",
    "    ),\n",
    "    row=1, col=1\n",
    ")\n",
    "\n",
    "# Vạch ngưỡng cảnh báo 85°C & nguy hiểm 95°C\n",
    "fig_sec.add_hline(y=85, line_dash=\"dash\", line_color=\"#D97706\", annotation_text=\"Cảnh báo (85°C)\", row=1, col=1)\n",
    "fig_sec.add_hline(y=95, line_dash=\"dot\", line_color=\"#DC2626\", annotation_text=\"Nguy hiểm (95°C)\", row=1, col=1)\n",
    "\n",
    "# Điểm Pop-out đỏ tiền chú ý cho các điểm vi phạm ngưỡng 95°C\n",
    "crit_pts = clean_df[clean_df['OT'] >= 95.0]\n",
    "fig_sec.add_trace(\n",
    "    go.Scatter(\n",
    "        x=crit_pts['date'], y=crit_pts['OT'],\n",
    "        mode='markers', marker=dict(color='#DC2626', size=6),\n",
    "        name='Vi phạm >= 95°C'\n",
    "    ),\n",
    "    row=1, col=1\n",
    ")\n",
    "\n",
    "# 2. Barcode Anomaly Strip dọc trục thời gian\n",
    "fig_sec.add_trace(\n",
    "    go.Scatter(\n",
    "        x=crit_pts['date'], y=np.ones(len(crit_pts)),\n",
    "        mode='markers',\n",
    "        marker=dict(symbol='line-ns', color='#DC2626', size=16, line=dict(width=2)),\n",
    "        name='Vị trí Sự cố (Rug Strip)'\n",
    "    ),\n",
    "    row=2, col=1\n",
    ")\n",
    "\n",
    "fig_sec.update_layout(\n",
    "    template='plotly_white',\n",
    "    height=600,\n",
    "    yaxis=dict(title='Nhiệt độ (°C)'),\n",
    "    yaxis2=dict(showticklabels=False, title='Sự cố'),\n",
    "    xaxis2=dict(title='Thời gian (Timestamp)')\n",
    ")\n",
    "fig_sec.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Khám phá 2: Đường cong Hồi quy phi tuyến LOESS (Precomputed LOESS Curve)\n",
    "sub_loess = loess_df[\n",
    "    (loess_df['season'] == 'Summer') &\n",
    "    (loess_df['feature'] == 'HUFL') &\n",
    "    (loess_df['thermal_regime'] == 'All')\n",
    "]\n",
    "\n",
    "fig_loess = px.line(\n",
    "    sub_loess,\n",
    "    x='feature_val',\n",
    "    y='loess_ot',\n",
    "    color_discrete_sequence=['#DC2626'],\n",
    "    labels={\n",
    "        'feature_val': 'Công suất tải Cao áp Hữu ích HUFL (MW)',\n",
    "        'loess_ot': 'Nhiệt độ dầu làm mịn LOESS (°C)'\n",
    "    },\n",
    "    title='<b>Action Title:</b> Mức gia tăng nhiệt độ dầu bắt đầu tăng tốc đột biến khi công suất tải Cao áp Hữu ích (HUFL) vượt mốc 40 MW'\n",
    ")\n",
    "\n",
    "fig_loess.update_layout(template='plotly_white')\n",
    "fig_loess.show()"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}